# signalign corpus factory — Colab runner

Fetches CC-BY / CC-BY-SA vocal tracks **with lyrics** from the Jamendo API,
runs the signalign dataset factory (Demucs → VAD → lyrics-informed forced
alignment → calibrated confidence gate), and saves gated, schema-valid
aligned segments.

**Runtime → Change runtime type → T4 GPU** before running.

Secrets needed (Colab left sidebar 🔑):
- `JAMENDO_CLIENT_ID` — free at https://devportal.jamendo.com
- `HF_TOKEN` — optional, for uploading results to Hugging Face

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/Alcadramin/signalign.git
%cd signalign
!pip install -q demucs faster-whisper silero-vad soundfile

In [ ]:
import torch, torchaudio
assert torch.cuda.is_available(), 'enable the T4 runtime'
assert hasattr(torchaudio.functional, 'forced_align'), 'torchaudio too old'
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__)

In [ ]:
from google.colab import userdata
CLIENT_ID = userdata.get('JAMENDO_CLIENT_ID')
MAX_TRACKS = 200
!python scripts/fetch_jamendo.py --client-id {CLIENT_ID} --max-tracks {MAX_TRACKS}

In [ ]:
config = '''
[input]
audio_dir = "data/jamendo_corpus/audio"
lyrics_dir = "data/jamendo_corpus/lyrics"
licenses_csv = "data/jamendo_corpus/tracks.csv"
source = "jamendo"
license = "unknown"

[output]
dir = "out/corpus"
'''
open('corpus.toml', 'w').write(config)
!python pipeline/run.py --config corpus.toml

In [ ]:
import json
for pile in ['keep', 'hard']:
    try:
        records = [json.loads(l) for l in open(f'out/corpus/{pile}.jsonl')]
        words = sum(len(r['words']) for r in records)
        hours = sum(r['duration'] for r in records) / 3600
        print(f'{pile}: {len(records)} segments, {words} words, {hours:.1f}h audio')
    except FileNotFoundError:
        print(pile, '- none')

## Upload results to Hugging Face

Uploads manifests + clips to a **private staging** dataset repo. The
public `signalign-corpus` release happens after human spot-check.

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
REPO = 'Alcadramin/signalign-corpus-staging'
!hf repo create {REPO} --repo-type dataset --private -y 2>/dev/null || true
!cp data/jamendo_corpus/tracks.csv out/corpus/
!hf upload {REPO} out/corpus . --repo-type dataset --commit-message 'colab factory run'